# OT Traffic Adaptation Strategy

This notebook demonstrates the methodology for adapting the NSL-KDD baseline intrusion detection model to live Operational Technology (OT) and SCADA environments in digitalized mining operations.

### Phase 1: Data Collection Pipeline

Our data acquisition strategy deploys mirroring taps at pilot mineral resource extraction plants. Network packets are captured on cloud-based AWS EC2 sniffer nodes. The raw PCAP stream is processed in real time by CICFlowMeter to output bidirectional flow CSV logs.

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from src.data.ot_collector import OTTrafficCollector
from src.data.swat import SWaTLoader
from src.optimization.bwoa import BinaryWhaleOptimizer
from src.optimization.fitness import FeatureFitnessEvaluator

# Simulate OT traffic feature generation
collector = OTTrafficCollector(interface="eth1", output_dir="data/raw/")
mock_csv = collector.capture(duration_seconds=5)

raw_flow_df = collector.load_captured(mock_csv)
print("Raw CICFlowMeter Output Columns (First 5 Rows):")
print(raw_flow_df.head())

Phase 1: Starting packet sniff on eth1 for 5 seconds.
Sniff command failed or dumpcap not present: [WinError 2] The system cannot find the file specified. Creating mock PCAP file.
Phase 1: Extracting traffic flow statistics via CICFlowMeter.
CICFlowMeter run failed: [WinError 2] The system cannot find the file specified. Creating mock flow CSV output.
Raw CICFlowMeter Output Columns (First 5 Rows):
   Flow Duration  Total Fwd Packets  ...  Flow Bytes/s  Flow Packets/s
0      37.454012          95.071431  ...     86.617615       60.111501
1      70.807258           2.058449  ...     30.424224       52.475643
2      43.194502          29.122914  ...     78.517596       19.967378
3      51.423444          59.241457  ...     96.563203       80.839735
4      30.461377           9.767211  ...     90.932040       25.877998

[5 rows x 9 columns]


In [2]:
# Show feature mapping from CICFlowMeter to NSL-KDD equivalents
mapped_ot_df = collector.align_features_to_nslkdd(raw_flow_df)
print(f"Aligned OT DataFrame shape: {mapped_ot_df.shape}")
print("Aligned OT DataFrame Features (First 5 Columns):")
print(mapped_ot_df.iloc[:, :5].head())

Aligned OT DataFrame shape: (50, 41)
Aligned OT DataFrame Features (First 5 Columns):
    duration  protocol_type  service  flag  src_bytes
0  37.454012            1.0      1.0   1.0  59.865848
1  70.807258            1.0      1.0   1.0  83.244264
2  43.194502            1.0      1.0   1.0  13.949386
3  51.423444            1.0      1.0   1.0  60.754485
4  30.461377            1.0      1.0   1.0  44.015249


In [3]:
# Demonstrate BWOA re-running on OT feature space
print("Running BWOA for features selection on aligned OT dataset...")
X_ot = mapped_ot_df.values
y_ot = np.random.randint(0, 2, size=(len(X_ot),))  # Mock targets

evaluator = FeatureFitnessEvaluator(alpha=0.88)
optimizer = BinaryWhaleOptimizer(
    n_agents=5,
    n_features=X_ot.shape[1],
    max_iter=2,
    fitness_fn=evaluator.calculate_fitness
)

best_ot_mask, ot_history = optimizer.optimize(X_ot, y_ot, X_ot, y_ot)
print(f"Selected {np.sum(best_ot_mask)} optimal features for the OT domain.")

Running BWOA for features selection on aligned OT dataset...
Iteration 1/2 - Best Fitness: 0.06150
Iteration 2/2 - Best Fitness: 0.03809
Selected 7 optimal features for the OT domain.


In [4]:
# Demonstrate transfer learning strategy (freeze CNN layers, retrain LSTM)
print("Loading NSL-KDD baseline model...")
try:
    base_model = tf.keras.models.load_model("models/cnn_lstm_nslkdd_baseline.keras")
except Exception:
    # Rebuild a dummy model if file is absent
    from src.models.cnn_lstm import build_cnn_lstm
    base_model = build_cnn_lstm(input_shape=(1, 41), n_classes=5)

print("\nOriginal Model Layers trainability status:")
for layer in base_model.layers:
    print(f"  Layer {layer.name} : Trainable = {layer.trainable}")

# Freeze Conv1D feature extractors
print("\nFreezing spatial extraction blocks (Conv1D) for domain transfer...")
for layer in base_model.layers:
    if "conv1d" in layer.name or "batch_normalization" in layer.name:
        layer.trainable = False

print("\nUpdated Model Layers trainability status:")
for layer in base_model.layers:
    print(f"  Layer {layer.name} : Trainable = {layer.trainable}")

Loading NSL-KDD baseline model...

Original Model Layers trainability status:
  Layer input_layer : Trainable = True
  Layer conv1d : Trainable = True
  Layer batch_normalization : Trainable = True
  Layer max_pooling1d : Trainable = True
  Layer dropout : Trainable = True
  Layer conv1d_1 : Trainable = True
  Layer batch_normalization_1 : Trainable = True
  Layer max_pooling1d_1 : Trainable = True
  Layer dropout_1 : Trainable = True
  Layer lstm : Trainable = True
  Layer dropout_2 : Trainable = True
  Layer dense : Trainable = True

Freezing spatial extraction blocks (Conv1D) for domain transfer...

Updated Model Layers trainability status:
  Layer input_layer : Trainable = True
  Layer conv1d : Trainable = False
  Layer batch_normalization : Trainable = False
  Layer max_pooling1d : Trainable = True
  Layer dropout : Trainable = True
  Layer conv1d_1 : Trainable = False
  Layer batch_normalization_1 : Trainable = False
  Layer max_pooling1d_1 : Trainable = True
  Layer dropout_1 : 

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 15 variables whereas the saved optimizer has 28 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### Domain Transfer Methodology Summary

This methodology forms the core of Phase 2. By freezing the spatial-pattern extraction layers (Conv1D) trained on broad baseline datasets, we preserve generic network anomaly features. Retraining only the recurrent (LSTM) and classification dense layers on target OT flow statistics minimizes computational cost, allowing local adaptations on resource-constrained deployment nodes.